# Project in Advanced Spatial Analysis GEOG3300

Crop type classification of an agricultural dataset in Germany

This is the project of Sylvia Ackermann (23888335).



## Setup
Download data

In [13]:
import os

if "germany_aoi" not in os.listdir(os.getcwd()):
    os.system('unzip "germany_aoi.zip"')

Install required packages

In [ ]:
!pip install geopandas
!pip install pyarrow
!pip install mapclassify
!pip install rasterio
!pip install pystac
!pip install pystac_client
!pip install planetary_computer
!pip install rasterstats
!pip install math

ERROR: Could not find a version that satisfies the requirement math (from versions: none)
ERROR: No matching distribution found for math


Import libraries

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import json
import os
import folium
import plotly.express as px
import plotly.io as pio
import pprint
from pystac.extensions.eo import EOExtension as eo
import pystac_client
import planetary_computer as pc
import rasterio
from rasterio import windows
from rasterio import features
from rasterio import warp
from rasterstats import zonal_stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GroupShuffleSplit
from sklearn.inspection import permutation_importance
from sklearn import tree
from sklearn import svm
import math


# setup renderer
pio.renderers.default = "colab"

## Task 1: Data import

In this task files of polygon points, which describe field boundaries, are saved as Geodataframes. First the path to the file is saved as a variable. Then the file is read to a Geodataframe. To check for the sucess the head() function is executed for the Geodataframe to display it's first rows on the screen.

Area 1

In [ ]:
print(os.getcwd())

/content


In [16]:
area1_path = os.path.join(os.getcwd(),"drive/MyDrive", "germany_aoi", "germany_aoi_1.gpkg")
print("path to file:", area1_path)

# open the file and save as GeoDataFrame
area1_gdf = gpd.read_file(area1_path)
area1_gdf.head()

ERROR:fiona._env:/content/drive/MyDrive/germany_aoi/germany_aoi_1.gpkg: No such file or directory


path to file: /content/drive/MyDrive/germany_aoi/germany_aoi_1.gpkg


DriverError: /content/drive/MyDrive/germany_aoi/germany_aoi_1.gpkg: No such file or directory

Area 2

In [ ]:
area2_path = os.path.join(os.getcwd(),"drive/MyDrive", "germany_aoi", "germany_aoi_2.shp")
print("path to file:", area2_path)

# open the file and save as GeoDataFrame
area2_gdf = gpd.read_file(area2_path)
area2_gdf.head()

Area 3

In [ ]:
area3_path = os.path.join(os.getcwd(),"drive/MyDrive", "germany_aoi", "germany_aoi_3.shp")
print("path to file:", area3_path)

# open the file and save as GeoDataFrame
area3_gdf = gpd.read_file(area3_path)
area3_gdf.head()

Area 4

In [ ]:
area4_path = os.path.join(os.getcwd(),"drive/MyDrive", "germany_aoi", "germany_aoi_4.gpkg")
print("path to file:", area4_path)

# open the file and save as GeoDataFrame
area4_gdf = gpd.read_file(area4_path)
area4_gdf.head()

## Task 2: Data Visualization

I have chosen the following color scheme to visualize the crop types:

*   Wheat - darkgoldenrod
*   Rye - saddlebrown
*   Barley - bisque
*   Oats - red
*   Corn - orange
*   Oil seeds - yellow
*   Root Crops - indigo
*   Meadows - yellowgreen
*   Forage crops - lightcoral

I chose these colors because these colors can intuitively be matched to the crop type because the colors match the look of the crop types. At the same time I chose colors, which can easily be distinguished from each other.

As basemap I chose the "CartoDB positron" map. This map is rich in detail but is dealing as a good background with light colors, so the colored field polygones pop out well.

In [ ]:
area1_gdf.explore("crop_name",["orange","lightcoral","yellowgreen","yellow","darkgoldenrod"], tiles="CartoDB positron")


In [ ]:
area2_gdf.explore("crop_name", ["bisque", "orange", "yellowgreen", "red", "yellow", "saddlebrown", "darkgoldenrod"], tiles="CartoDB positron")

In [ ]:
area3_gdf.explore("crop_name",["bisque","orange", "lightcoral","yellowgreen", "red", "yellow", "indigo", "saddlebrown", "darkgoldenrod"], tiles="CartoDB positron")

In [ ]:
area4_gdf.explore("crop_name",["bisque","orange","lightcoral","yellowgreen","red","yellow","saddlebrown","darkgoldenrod"], tiles="CartoDB positron")


## Task 3 - Data exploration


1) Printing summary table with count of number of observations of each crop type

2) Printing crs property of each of the GeoDataFrames

3) Identifying the EPSG string that describes the GeoDataFrame's coordinate reference system

Area 1

In [ ]:
# find number of measurements for each crop type
cropTypeGroups = area1_gdf.groupby(["crop_name"])
numCropTypes = cropTypeGroups.size()
print(numCropTypes)

In [ ]:
# print crs property
area1_gdf.crs

In [ ]:
# print EPSG string
area1_gdf.crs.to_epsg()

Area 2

In [ ]:
# find number of measurements for each crop type
cropTypeGroups = area2_gdf.groupby(["crop_name"])
numCropTypes = cropTypeGroups.size()
print(numCropTypes)

In [ ]:
# print crs property
area2_gdf.crs

In [ ]:
# print EPSG string
area2_gdf.crs.to_epsg()

Area 3

In [ ]:
# find number of measurements for each crop type
cropTypeGroups = area3_gdf.groupby(["crop_name"])
numCropTypes = cropTypeGroups.size()
print(numCropTypes)

In [ ]:
# print crs property
area3_gdf.crs

In [ ]:
# print EPSG string
area3_gdf.crs.to_epsg()

Area 4

In [ ]:
# find number of measurements for each crop type
cropTypeGroups = area4_gdf.groupby(["crop_name"])
numCropTypes = cropTypeGroups.size()
print(numCropTypes)

In [ ]:
# print crs property
area4_gdf.crs

In [ ]:
# print EPSG string
area4_gdf.crs.to_epsg()

## Task 4 - Data transformation

create smallest bounding box geometry around all the crop field polygons of one area

set coordinate reference system property (crs) of the bounding box to match the GeoDataFrame

Area 1

In [ ]:
# combine the GeoDataFrame of many polygon objects into a single multipolygon object
area1_gdf_union = gpd.GeoSeries(area1_gdf["geometry"].unary_union)

# create bounding box around whole area
area1_boundingBox = area1_gdf_union.envelope.set_crs("EPSG:25833")

# check if bounding box is in the right location
area1_boundingBox.explore()

Area 2

In [ ]:
# combine the GeoDataFrame of many polygon objects into a single multipolygon object
area2_gdf_union = gpd.GeoSeries(area2_gdf["geometry"].unary_union)

# create bounding box around whole area
area2_boundingBox = area2_gdf_union.envelope.set_crs("EPSG:25833")

# check if bounding box is in the right location
area2_boundingBox.explore()

Area 3

In [ ]:
# combine the GeoDataFrame of many polygon objects into a single multipolygon object
area3_gdf_union = gpd.GeoSeries(area3_gdf["geometry"].unary_union)

# create bounding box around whole area
area3_boundingBox = area3_gdf_union.envelope.set_crs("EPSG:25833")

# check if bounding box is in the right location
area3_boundingBox.explore()

Area 4

In [ ]:
# combine the GeoDataFrame of many polygon objects into a single multipolygon object
area4_gdf_union = gpd.GeoSeries(area4_gdf["geometry"].unary_union)

# create bounding box around whole area
area4_boundingBox = area4_gdf_union.envelope.set_crs("EPSG:25833")

# check if bounding box is in the right location
area4_boundingBox.explore()

**Why does the unary_union operation need to be performed before computing the envelope?**

The unary_union operation combines all polygones of the different crop fields into one multi-polygone object. This needs to be done before computing the envelope to create one bounding box around the whole multi-polygone containing all crop fields within that area.

If the envelope computation would be done for the GeoDataFrame containing all single field polygones without using the unary_union operation, multiple bounding boxes around each single field would be computed.


## Task 5 - Data transformation

Transformation of coordinate reference system

From: EPSG:25833
- Name: ETRS89 / UTM zone 33N
- cartesian system with axes Easting (metre) and Northing (metre)

---
To: EPSG:4326
- Name: WGS 84
- ellipsoidal system with axes Geodetic latitude (degree) and Geodetic longitude (degree)



Area 1

In [ ]:
# transform bounding box into WGS84 reference system
area1_boundingBox = area1_boundingBox.to_crs("EPSG:4326")
# check if crs-transformation was succesful
area1_boundingBox.crs

Area 2

In [ ]:
# transform bounding box into WGS84 reference system
area2_boundingBox = area2_boundingBox.to_crs("EPSG:4326")
# check if crs-transformation was succesful
area2_boundingBox.crs

Area 3

In [ ]:
# transform bounding box into WGS84 reference system
area3_boundingBox = area3_boundingBox.to_crs("EPSG:4326")
# check if crs-transformation was succesful
area3_boundingBox.crs

Area 4

In [ ]:
# transform bounding box into WGS84 reference system
area4_boundingBox = area4_boundingBox.to_crs("EPSG:4326")
# check if crs-transformation was succesful
area4_boundingBox.crs

## Task 6 - Machine Learning Model

The geospatial dataset used for this task are images from the Sentinel 2 satellite. The dataset will be downloaded from https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a#overview. This dataset fits the task because it provides measurements in all channels required for this classification task including the visual wavelengths and infrared images. Furthermore the images have a ground sampling distance (GSD) of 10 m, which is small enough in comparison to the sizes of the classified crop-fields.

From this dataset in particular the blue, green, red, rededge and near infrared channels will be used. The data of these channels is imported in task 7.

The input features of the Machine Learning Model, called predictors, are a combination of the reflectance values of the different channels and several vegetation indices, which are derived from the reflectance values.
The vegetation indices are chosen from this selection of widely used optical vegetation indices: https://www.nature.com/articles/s43017-022-00298-5/tables/1. The formulas of the vegetation indices are derived from the table as well.

Apart from that, a helpful overview of the nine most useful vegetation indices for crop type classification can be found in the article "Extending Crop Type Reference Data Using a Phenology-Based Approach". (https://www.frontiersin.org/articles/10.3389/fsufs.2020.00099/full)

After inspiration from both articles the following vegetation indices are chosen for this task of crop type classification:

* Difference Vegetation Index (DVI)
* Normalized Difference Vegetation Index (NDVI)
* Green Normalized Difference Vegetation Index (GNDVI)
* Modified Red Edge Simple Ratio Index (MSR)
* Enhanced Vegetation Index (EVI)
* Modified Soil Adjusted Vegetation Index (MSAVI)
* Green chromatic coordinate (GCC)

The choice of vegetation indices has been made by finding indices, which are listed in both articles, which indicates their importance. Furthermore the list of indices is extended to incorporate reflectance values of the whole spectrum into the calculation of the vegetation indices.

In order to generate an optimal machine learning model, training with different subsets of indices is done. Evaluations of the test results show, that the Difference Vegatation Index and the Modified Soil Adjusted Vegetation Index do not improve the model quality. In fact the accuracy is increasing, once these indices are removed from the list of predictors, so I decided to remove them from the list.

## Task 7 - Data import

import of the geospatial datasets that will be used as input features
(predictors)



In [ ]:
# Data access
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

The time period of July was chosen according to the European Crop Calendar:
https://ipad.fas.usda.gov/rssiws/al/crop_calendar/europe.aspx

The calendar shows that most crop types are harvested in July. So I decided to use satellite images from the month of June to evaluate the fields shortly before harvesting.



In [ ]:
# define time of interest
time_of_interest = "2018-06-01/2018-06-30"  # time: June 2018

### Area 1

In [ ]:
# convert GeoSeries bounding-box-polygone into json
area_json = json.loads(area1_boundingBox.to_json())

# store relevant entries (geometry) in dictionary
area1_geometry = dict(area_json["features"][0])["geometry"]
print(area1_geometry)

In [ ]:
# search for data within catalog
search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=area1_geometry,
    datetime=time_of_interest,
    query={"eo:cloud_cover": {"lt": 10}},
)
# Check how many items were returned to see that download was sucessful
items = search.item_collection()
print(f"Returned {len(items)} Items")

In [ ]:
# find the image with minimal cloud cover
cloud_cover = []
for i in items:
    cloud_cover.append(eo.ext(i).cloud_cover)
min_cloud_cover = min(cloud_cover)
min_cloud_cover_idx = cloud_cover.index(min_cloud_cover)
least_cloudy_image = items[min_cloud_cover_idx]

In [ ]:
# print assets properties of STAC Item
least_cloudy_image.assets.keys()

In [ ]:
# check for wanted assets
least_cloudy_image.assets["B02"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B02"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area1_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area1_blue_band_data = ds.read(1, window=area_window) # read data from the window
    out_meta_area1 = ds.meta  # copy of metadata is needed later

# visualize image
px.imshow(area1_blue_band_data, color_continuous_scale="Blues")

In [ ]:
out_meta_area1

In [ ]:
# update out meta for this area
out_meta_area1["dtype"] = "float32"
out_meta_area1["height"] = area1_blue_band_data.shape[0]
out_meta_area1["width"] = area1_blue_band_data.shape[1]
out_meta_area1["transform"] = ds.window_transform(area_window)
out_meta_area1

In [ ]:
area1_blue_band_data.shape

In [ ]:
least_cloudy_image.assets["B03"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B03"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area1_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area1_green_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area1_green_band_data, color_continuous_scale="Greens")

In [ ]:
least_cloudy_image.assets["B04"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B04"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area1_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area1_red_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area1_red_band_data, color_continuous_scale="Reds")

In [ ]:
least_cloudy_image.assets["B06"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B06"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area1_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area1_red_edge_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area1_red_edge_band_data, color_continuous_scale="Reds")

In [ ]:
least_cloudy_image.assets["B08"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B08"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area1_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area1_nir_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area1_nir_band_data, color_continuous_scale="viridis")

### Area 2

In [ ]:
# convert GeoSeries bounding-box-polygone into json
area_json = json.loads(area2_boundingBox.to_json())

# store relevant entries (geometry) in dictionary
area2_geometry = dict(area_json["features"][0])["geometry"]
print(area2_geometry)

In [ ]:
# search for data within catalog
search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=area2_geometry,
    datetime=time_of_interest,
    query={"eo:cloud_cover": {"lt": 10}},
)
# Check how many items were returned to see that download was sucessful
items = search.item_collection()
print(f"Returned {len(items)} Items")

In [ ]:
# find the image with minimal cloud cover
cloud_cover = []
for i in items:
    cloud_cover.append(eo.ext(i).cloud_cover)
min_cloud_cover = min(cloud_cover)
min_cloud_cover_idx = cloud_cover.index(min_cloud_cover)
least_cloudy_image = items[min_cloud_cover_idx]

In [ ]:
# print assets properties of STAC Item
least_cloudy_image.assets.keys()

In [ ]:
# check for wanted assets
least_cloudy_image.assets["B02"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B02"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area2_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area2_blue_band_data = ds.read(1, window=area_window) # read data from the window
    out_meta_area2 = ds.meta  # copy of metadata is needed later

# visualize image
px.imshow(area2_blue_band_data, color_continuous_scale="blues")

In [ ]:
out_meta_area2

In [ ]:
# update out meta for this area
out_meta_area2["dtype"] = "float32"
out_meta_area2["height"] = area1_blue_band_data.shape[0]
out_meta_area2["width"] = area1_blue_band_data.shape[1]
out_meta_area2["transform"] = ds.window_transform(area_window)
out_meta_area2

In [ ]:
least_cloudy_image.assets["B03"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B03"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area2_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area2_green_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area2_green_band_data, color_continuous_scale="greens")

In [ ]:
least_cloudy_image.assets["B04"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B04"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area2_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area2_red_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area2_red_band_data, color_continuous_scale="reds")

In [ ]:
least_cloudy_image.assets["B06"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B06"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area2_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area2_red_edge_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area2_red_edge_band_data, color_continuous_scale="Reds")

In [ ]:
least_cloudy_image.assets["B08"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B08"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area2_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area2_nir_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area2_nir_band_data, color_continuous_scale="viridis")

### Area 3

In [ ]:
# convert GeoSeries bounding-box-polygone into json
area_json = json.loads(area3_boundingBox.to_json())

# store relevant entries (geometry) in dictionary
area3_geometry = dict(area_json["features"][0])["geometry"]
print(area3_geometry)

In [ ]:
# search for data within catalog
search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=area3_geometry,
    datetime=time_of_interest,
    query={"eo:cloud_cover": {"lt": 10}},
)
# Check how many items were returned to see that download was sucessful
items = search.item_collection()
print(f"Returned {len(items)} Items")

In [ ]:
# find the image with minimal cloud cover
cloud_cover = []
for i in items:
    cloud_cover.append(eo.ext(i).cloud_cover)
min_cloud_cover = min(cloud_cover)
min_cloud_cover_idx = cloud_cover.index(min_cloud_cover)
least_cloudy_image = items[min_cloud_cover_idx]

In [ ]:
# print assets properties of STAC Item
least_cloudy_image.assets.keys()

In [ ]:
# check for wanted assets
least_cloudy_image.assets["B02"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B02"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area3_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area3_blue_band_data = ds.read(1, window=area_window) # read data from the window
    out_meta_area3 = ds.meta  # copy of metadata is needed later

# visualize image
px.imshow(area3_blue_band_data, color_continuous_scale="blues")

In [ ]:
out_meta_area3

In [ ]:
# update out meta for this area
out_meta_area3["dtype"] = "float32"
out_meta_area3["height"] = area1_blue_band_data.shape[0]
out_meta_area3["width"] = area1_blue_band_data.shape[1]
out_meta_area3["transform"] = ds.window_transform(area_window)
out_meta_area3

In [ ]:
# check for wanted assets
least_cloudy_image.assets["B03"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B03"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area3_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area3_green_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area3_green_band_data, color_continuous_scale="greens")

In [ ]:
# check for wanted assets
least_cloudy_image.assets["B04"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B04"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area3_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area3_red_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area3_red_band_data, color_continuous_scale="reds")

In [ ]:
least_cloudy_image.assets["B06"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B06"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area3_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area3_red_edge_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area3_red_edge_band_data, color_continuous_scale="Reds")

In [ ]:
# check for wanted assets
least_cloudy_image.assets["B08"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B08"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area3_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area3_nir_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area3_nir_band_data, color_continuous_scale="viridis")

### Area 4

In [ ]:
# convert GeoSeries bounding-box-polygone into json
area_json = json.loads(area4_boundingBox.to_json())

# store relevant entries (geometry) in dictionary
area4_geometry = dict(area_json["features"][0])["geometry"]
print(area4_geometry)

In [ ]:
# search for data within catalog
search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=area4_geometry,
    datetime=time_of_interest,
    query={"eo:cloud_cover": {"lt": 10}},
)
# Check how many items were returned to see that download was sucessful
items = search.item_collection()
print(f"Returned {len(items)} Items")

In [ ]:
# find the image with minimal cloud cover
cloud_cover = []
for i in items:
    cloud_cover.append(eo.ext(i).cloud_cover)
min_cloud_cover = min(cloud_cover)
min_cloud_cover_idx = cloud_cover.index(min_cloud_cover)
least_cloudy_image = items[min_cloud_cover_idx]

In [ ]:
# print assets properties of STAC Item
least_cloudy_image.assets.keys()

In [ ]:
# check for wanted assets
least_cloudy_image.assets["B02"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B02"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area4_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area4_blue_band_data = ds.read(1, window=area_window) # read data from the window
    out_meta_area4 = ds.meta  # copy of metadata is needed later

# visualize image
px.imshow(area4_blue_band_data, color_continuous_scale="blues")

In [ ]:
out_meta_area4

In [ ]:
# update out meta for this area
out_meta_area4["dtype"] = "float32"
out_meta_area4["height"] = area1_blue_band_data.shape[0]
out_meta_area4["width"] = area1_blue_band_data.shape[1]
out_meta_area4["transform"] = ds.window_transform(area_window)
out_meta_area4

In [ ]:
# check for wanted assets
least_cloudy_image.assets["B03"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B03"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area4_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area4_green_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area4_green_band_data, color_continuous_scale="greens")

In [ ]:
# check for wanted assets
least_cloudy_image.assets["B04"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B04"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area4_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area4_red_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area4_red_band_data, color_continuous_scale="reds")

In [ ]:
least_cloudy_image.assets["B06"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B06"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area4_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area4_red_edge_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area4_red_edge_band_data, color_continuous_scale="Reds")

In [ ]:
# check for wanted assets
least_cloudy_image.assets["B08"]

In [ ]:
# create signed link to requested band
band_href = pc.sign(least_cloudy_image.assets["B08"].href)
# open a connection to the COG using its signed link
with rasterio.open(band_href) as ds:
    area_bounds = features.bounds(area4_boundingBox[0])  # bounding box
    warped_area_bounds = warp.transform_bounds("epsg:4326", ds.crs, *area_bounds)  # CRS data
    area_window = windows.from_bounds(transform=ds.transform, *warped_area_bounds) # window object
    area4_nir_band_data = ds.read(1, window=area_window) # read data from the window

# visualize image
px.imshow(area4_nir_band_data, color_continuous_scale="viridis")

## Task 8 - data exploration and transformation

transformation of geospatial data into pandas DataFrame with columns for each input (predictor) and rows for each area

### Compute predictors

In [ ]:
area1_data = [area1_blue_band_data, area1_green_band_data, area1_red_band_data, area1_red_edge_band_data, area1_nir_band_data]
area2_data = [area2_blue_band_data, area2_green_band_data, area2_red_band_data, area2_red_edge_band_data, area2_nir_band_data]
area3_data = [area3_blue_band_data, area3_green_band_data, area3_red_band_data, area3_red_edge_band_data, area3_nir_band_data]
area4_data = [area4_blue_band_data, area4_green_band_data, area4_red_band_data, area4_red_edge_band_data, area4_nir_band_data]
predictor_names = ["blue","green","red","red edge","nir"]

In [ ]:
# Difference Vegetation Index (DVI)
def DVI(data):
  red = data[2].astype("float32")
  nir = data[4].astype("float32")
  return nir-red

In [ ]:
# Normalized Difference Vegetation Index
def NDVI(data):
  red = data[2].astype("float32")
  nir = data[4].astype("float32")
  return (nir - red) / (nir + red)

In [ ]:
# Green Normalized Difference Vegetation Index (GNDVI)
def GNDVI(data):
  green = data[1].astype("float32")
  nir = data[4].astype("float32")
  return (nir - green) / (nir + green)

In [ ]:
# Modified Red Edge Simple Ratio Index (MSR)
def MSR(data):
  red = data[2].astype("float32")
  nir = data[4].astype("float32")
  return (nir/(red-1))/np.sqrt(nir/(red+1))

In [ ]:
# Enhanced Vegetation Index (2 band version) (EVI)
def EVI(data):
  red = data[2].astype("float32")
  nir = data[4].astype("float32")
  return 2.5*((nir-red)/(nir+2.4*red+1))

In [ ]:
# Modified Soil Adjusted Vegetation Index (MSAVI)
def MSAVI(data):
  red = data[2].astype("float32")
  nir = data[4].astype("float32")
  return (2 * nir + 1 - np.sqrt(np.square(2 * nir + 1) - 8*(nir-red)))/2

In [ ]:
# Green chromatic coordinate (GCC)
def GCC(data):
  blue = data[0].astype("float32")
  green = data[1].astype("float32")
  red = data[2].astype("float32")
  return green/(red+green+blue)

In [ ]:
# creata lists including data of all areas
d = area1_data
area1_data += [ NDVI(d), GNDVI(d), MSR(d), EVI(d), GCC(d)]
d = area2_data
area2_data += [ NDVI(d), GNDVI(d), MSR(d), EVI(d),  GCC(d)]
d = area3_data
area3_data += [ NDVI(d), GNDVI(d), MSR(d), EVI(d),  GCC(d)]
d = area4_data
area4_data += [ NDVI(d), GNDVI(d), MSR(d), EVI(d), GCC(d)]
predictor_names += ["NDVI","GNDVI","MSR","EVI","GCC"]

In [ ]:
print(predictor_names)

In [ ]:
# plot one image to check if calculation worked
# EVI[0,2], DVI [0,5000], GNDVI[0,1], MSR[1,6], NDVI[0,1], MSAVI[-0.2,0.8], GCC[0.3,0.6]
px.imshow(NDVI(area1_data), color_continuous_scale="BuGn")

### Generate combined GeoDataFrame

extract data within the area of the field polygones from the images containing the remote sensing data

In [ ]:
def calculateFieldValues(area_gdf,area_data, out_meta):
  area_gdf_tmp = area_gdf.to_crs(out_meta["crs"])
  area_gdf_data = area_gdf_tmp.copy()

  for i in range(len(area_data)):

    zstats = zonal_stats(
      area_gdf_tmp,
      area_data[i],
      affine=out_meta["transform"],
      stats=["mean"],
      all_touched=True
    )

    df_zstats = pd.DataFrame(zstats)
    area_gdf_data[predictor_names[i]] = df_zstats

  return area_gdf_data

In [ ]:
area1_gdf_data = calculateFieldValues(area1_gdf,area1_data,out_meta_area1)
area1_gdf_data['area'] = 1
area1_gdf_data.head()

In [ ]:
area2_gdf_data = calculateFieldValues(area2_gdf,area2_data,out_meta_area2)
area2_gdf_data['area'] = 2
area2_gdf_data.head()

In [ ]:
area3_gdf_data = calculateFieldValues(area3_gdf,area3_data,out_meta_area3)
area3_gdf_data['area'] = 3
area3_gdf_data.head()

In [ ]:
area4_gdf_data = calculateFieldValues(area4_gdf,area4_data,out_meta_area4)
area4_gdf_data['area'] = 4
area4_gdf_data.head()

In [ ]:
dfColumns = ["crop_id","crop_name","geometry","area"]+predictor_names
allFields_gdf = pd.concat([area1_gdf_data[dfColumns],area2_gdf_data[dfColumns],area3_gdf_data[dfColumns],area4_gdf_data[dfColumns]], axis=0)
allFields_gdf.shape

In [ ]:
allFields_gdf.head()

### Removing Outliers

display statistics per crop type for each predictor

In [ ]:
display(allFields_gdf[["crop_name"]+predictor_names].groupby(["crop_name"]).describe().T)

In [ ]:
# Crop Types and fields before outlier removal
cropType_names = ["Barley",	"Corn",	"Forage Crops",	"Meadows",	"Oats",	"Oil Seeds",	"Root Crops",	"Rye",	"Wheat"]
allFields_gdf_cleaned = gpd.GeoDataFrame()
count_cropTypes = allFields_gdf.groupby(["crop_name"]).count().shape[0]
print(f"BEFORE removing outliers {allFields_gdf.shape[0]} fields of {count_cropTypes} crop types are stored in the GeoDataFrame allFields_gdf \n")

# Remove outlier fields from GeoDataFrame
# Remove if at least one predictors value is further than 2 times the standard deviation from the mean
for current_cropType in cropType_names:
  df_crop = allFields_gdf.loc[allFields_gdf["crop_name"] == current_cropType, :]
  print(f"There are {df_crop.shape[0]} {current_cropType} rows BEFORE dropping outliers")

  for predictor in predictor_names:
    df_crop = df_crop.loc[(df_crop[predictor]-df_crop[predictor].mean()).abs() < (2*df_crop[predictor].std()), :]

  print(f"There are {df_crop.shape[0]} {current_cropType} rows AFTER dropping outliers")
  allFields_gdf_cleaned = pd.concat([allFields_gdf_cleaned,df_crop])
  print("----------------------------------------------------------")

count_cropTypes = allFields_gdf_cleaned.groupby(["crop_name"]).count().shape[0]
print(f"AFTER removing outliers {allFields_gdf_cleaned.shape[0]} fields of {count_cropTypes} crop types are stored in the GeoDataFrame allFields_gdf_cleaned")

In [ ]:
allFields_gdf_cleaned.head()

In [ ]:
display(allFields_gdf_cleaned[["crop_name"]+predictor_names].groupby(["crop_name"]).describe().T)

In this task outliers were removed from the GeoDataFrame of all field data.
This is done by looping over all crop types and predictors and finding those fields, where a predictor's value has a higher distance from the mean than 3 times the standard deviation. These fields are removed from the GeoDataFrame. It is assumed that these fields are not representative for their crop type.

Before removing the outliers 407 fields of 9 crop types were stored in the GeoDataFrame. After removing the outliers 398 fields of 8 crop types are stored in the GeoDataFrame. The crop type "Root Crops" was completely removed from the dataset. It's one sample field was no

## Task 9 - Visualisation of input features

To test the quality of the data multiple visualizations are created in this task.


* To determine the balance of observations across outcome features, a histogram of the crop types is created. The histogram visualizes the number of fields of each crop type in the dataset. Each crop type is visualized in a different color to make them easily distinguishable for the reader. A sign of a dataset of good quality is when there are enough samples of each crop type and their ammount is evenly distributed.

* Furthermore a histogram of each predictor-crop type-pair is generated. This shows the behavior of each predictor for all crop types and it can be checked if the outlier removal in task 8 was succesful.

* Lastly a histogram for each predictor is created, which includes all crop types in different colors. Therefore the facet_col parameter in the previous histogram is replaced with the color parameter. This way the values of one predictor for different crop types can be compared. A high difference in the predictor values for different crop types will make the crop types well distinguishable which will lead to good classification results



### Balance of observations across outcome features

In [ ]:
px.histogram(allFields_gdf_cleaned, x="crop_name", color="crop_name")

The dataset is containing many fields, which are labeled as Meadow. On the other hand it contains only a few fields labeled as Oats. This might lead to problems in the classification in the sense that fields will be more likely to be classified as Meadows, than as Oat fields.

Apart from these two edge cases fields of all other crop types appear with an equal quantity in the dataset, which should lead to good classification resuls once the network is trained.

### Check data cleaning

In [ ]:
for predictor in predictor_names:
  fig = px.histogram(
      data_frame=allFields_gdf_cleaned,
      x=predictor,
      facet_col="crop_name",
      marginal="box",
      hover_data=predictor_names)
  fig.show()

The visualization of the historgrams for all crop types and predictors shows a distribution of the data around the mean predictor value. For the predictors of some crop types a clear mean is recognizable, others are further distributed.

As far as one can tell from visual analyzation there are no significant outliers in the cleaned dataset.

### Relationship between features and outcome labels

In [ ]:
for predictor in predictor_names:
  fig = px.histogram(
      data_frame=allFields_gdf_cleaned,
      x=predictor,
      color="crop_name",
      marginal="box",
      hover_data=predictor_names)
  fig.show()

Visualizing the distribution of predictor values in one graph for all crop types shows the difference in predictor values for different crop types. Especially the visualizations for NIR- and NDVI-values show a completely different pattern than the visualizations for red, green and blue values. The combination of all these predictors will make the different crop types clearly distinguishable.

## Task 10 - Train machine learning model

create DataFrame of input data X and Series of output labels y

In [ ]:
allFields_gdf_cleaned.head()

In [ ]:
X = allFields_gdf_cleaned.drop(["crop_id", "crop_name", "geometry","area"], axis=1)
y = allFields_gdf_cleaned.loc[:, ["crop_name"]]

In [ ]:
X.head()

In [ ]:
y.head()

**Strategy to evaluate the model**

To evaluate the model a subset of the data will be used for testing. Therefore the dataset will be separated into a training set (80 %) and a test set (20 %). The training set will be used to train the parameters of the model and learn the different crop types. The test set will be used to evaluate the quality of the model.

In order to have an unbiased evaluation the function GroupShuffleSplit is used. This function splits the data into training and test set by avoiding spatial correlation between the training and test dataset. By looking at the histograms of training and test dataset dependent on the area, I could see that the algorithm declares the fields in area 3 as test dataset and the fields in the other areas as training dataset.

In [ ]:
rng = np.random.RandomState(0)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, train_size=.8, random_state=0)
groups = allFields_gdf_cleaned.loc[:, "area"]

for i, (train_index, test_index) in enumerate(gss.split(X, y, groups)):
    X_train = X.iloc[train_index, :]
    X_test = X.iloc[test_index, :]
    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]
    print(f"the size of the training set is {y_train.shape[0]}")
    print(f"the size of the test set is {y_test.shape[0]}")

In [ ]:
# px.histogram(y_train, x="crop_name", color="crop_name")

In [ ]:
# px.histogram(y_test, x="crop_name", color="crop_name")

train the model with the Random Forest classifier by creating an ensemble of 100 trees

In [ ]:
# create and train a random forests model
rf = RandomForestClassifier(n_estimators=100, random_state=rng)
rf.fit(X_train, y_train)

## Task 11 - Evaluate machine learning model

**Metrics used to assess model performance**

The classification report gives an overview of the quality of the model and its performance for the test dataset.
The classification report includes the following metrics:

* **precision**: the classifiers ability not to label something as positive when it is not
* **recall**: the ability of the classifier to capture all positive cases
* **f1-score**: the f1-score combines the recall and precision scores

The evaluation of all these metrics gives a valuable evaluation of the machine learning model. The height of the f1-score is a helpful indicator about the quality of the dataset in terms of recall and precision. If the f1-score is low it makes sense to look into the parameters recall and precision seperately to identify possible reasons.

Apart from the classification report the confusion matrix is generated. This matrix describes how many fields of each crop type are predicted in the right way. Ideally the highest number should be on the main diagonal of the matrix as a sign of a good model quality.

Lastly the machine learning model will be used to predict the crop types for all input predictors and fields to visualize the predicted crop types of all fields. As a comparison the true crop types for the fields in all areas are visualized as well (as in task 2). To evaluate the model specifically area 3, which contains the fields of the test dataset, needs to be reviewed. This area is located in the middle of all areas.



In [ ]:
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

Confusion Matrix

In [ ]:
labels = cropType_names
labels.remove("Root Crops")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(text_kw={"fontsize":10}, xticks_rotation="vertical")
plt.show()

In [ ]:
# px.histogram(y_test, x="crop_name", color="crop_name")

Visualize field crop types on map

In [ ]:
all_preds = rf.predict(X)
allFields_gdf_cleaned["predicted"] = all_preds

In [ ]:
colors = ["bisque","orange","lightcoral","yellowgreen","red","yellow","saddlebrown","darkgoldenrod"]

Predicted crop types

In [ ]:
allFields_gdf_cleaned.explore("predicted",colors, tiles="CartoDB positron",tooltip=["crop_name", "predicted"])

True crop types


In [ ]:
allFields_gdf_cleaned.explore("crop_name",colors, tiles="CartoDB positron")

Testing of different predictor combinations


accuracy of 0.44 with predictors Blue, Green, Red, NIR, NDVI

accuracy of 0.57 with predictors ['blue', 'green', 'red', 'red edge', 'nir', 'DVI', 'NDVI', 'GNDVI', 'MSR', 'EVI', 'MSAVI', 'GCC']

accuracy of 0.59 with predictors ['blue', 'green', 'red', 'red edge', 'nir', 'NDVI', 'GNDVI', 'MSR', 'EVI', 'MSAVI', 'GCC']

accuracy of 0.51 with predictors ['blue', 'green', 'red', 'red edge', 'nir', 'GNDVI', 'MSR', 'EVI', 'MSAVI', 'GCC']

accuracy of 0.50 with predictors ['blue', 'green', 'red', 'red edge', 'nir', 'NDVI', 'MSR', 'EVI', 'MSAVI', 'GCC']

accuracy of 0.54 with predictors ['blue', 'green', 'red', 'red edge', 'nir', 'NDVI', 'GNDVI', 'EVI', 'MSAVI', 'GCC']

accuracy of 0.52 with predictors ['blue', 'green', 'red', 'red edge', 'nir', 'NDVI', 'GNDVI', 'MSR', 'MSAVI', 'GCC']

accuracy of 0.61 with predictors ['blue', 'green', 'red', 'red edge', 'nir', 'NDVI', 'GNDVI', 'MSR', 'EVI', 'GCC']


**The models suitability**

The evaluatiion of my machine learning model shows that it can reach an overall accuracy of 61 %. Classifications work well for some crop types. The prediction of oil seeds is the only crop type with an accuracy and recall of 100 %. The classification of Barley is correct in most cases as well. On the other hand Oats receive an F1-score of 0 %, which means the classification is wrong in all test cases. Misclassifications haben in multiple cases, such as the classification of Forage Crops as Meadows.

i) *Improve the model's classification performance*

For better results more labeled field data need to be used for the training of the model. The amount of training data needs to be increased especially for crop types with only a few field samples, such as Oat fields for instance. Furthermore, the list of predictors can be extended with more vegetation indices. Extending the list of predictors by four new predictors, has increased the model's accuracy from 44 to 61 % already. This means a further extension of the predictors can improve the models performance even more.  

ii) *Adapt the model for deployment in new cropping landscapes*

In order to apply the machine learning model in new cropping landscapes, new training data needs to be used, which is representing the new cropping landscape. Especially for cropping landscapes such as Intercropping or Agroforestry it is important to use adapted training and testing data.

